# 04 · Agentic RAG

这一节参考课件 `RAG_theory` 的 Agentic RAG：
- **CRAG**：先评估检索质量，不够好就纠错/补检索
- **Adaptive-RAG**：先给 query 分流（简单走轻链路，复杂走重链路）
- **MemoRAG**：把高价值中间结果沉淀成 memory，后续优先复用

本 notebook 目标是“讲清楚控制闭环”，所以实现尽量精简直观：
- 仍然复用前面写好的 Chroma collection：`autel_annual_report_2024`
- 用 LLM 做路由/评估（可替换成规则/小模型）
- 每一步都打印 trace，便于课堂讲解
- 同时把关键步骤上报到 Langfuse，方便在可视化界面里看 agent 轨迹

> 依赖：先跑 `01_data_02_chunk_ingest.ipynb` 写入 `data/chroma`。


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Literal
from uuid import uuid4

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL")
chat_model = os.getenv("CHAT_MODEL")
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_base_url = os.getenv("LANGFUSE_BASE_URL")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_public_key, f"未加载 LANGFUSE_PUBLIC_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_secret_key, f"未加载 LANGFUSE_SECRET_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_base_url, f"未加载 LANGFUSE_BASE_URL（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
langfuse = get_client()
LF_USER_ID = os.getenv("LANGFUSE_USER_ID", "rag-notebook-user")
LF_SESSION_ID = f"agentic-rag-{uuid4().hex[:8]}"
LF_TAGS = ["rag_project", "agentic_rag", COLLECTION]


def lf_observe(name: str, as_type: str = "agent", **kwargs):
    """Langfuse v4: 返回 context manager，内部所有 CallbackHandler() 自动嵌套其下。"""
    return langfuse.start_as_current_observation(name=name, as_type=as_type, **kwargs)

def lf_config(step_name: str, tags: list[str] | None = None):
    """返回 LangChain config，自动嵌套在当前 observation context 下。"""
    return {
        "callbacks": [CallbackHandler()],
        "run_name": step_name,
    }


print(
    "ready:",
    COLLECTION,
    "embed:",
    embed_model,
    "chat:",
    chat_model,
    "env:",
    ENV_FILE,
    "langfuse_session:",
    LF_SESSION_ID,
)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_51732/247134885.py:65: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))


ready: autel_annual_report_2024 embed: Qwen/Qwen3-Embedding-8B chat: deepseek-ai/DeepSeek-V3.2 env: /Users/mengbai/Documents/AI-training/.env langfuse_session: agentic-rag-b6d95f8b


## 用户Query

In [5]:
Q0 = "道通2024年年报里，各产品利润率是多少？" 

In [6]:
# 工具：向量检索（带最小 trace）


def parse_json_object(raw: str, fallback: dict):
    text = (raw or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data
    except Exception:
        pass

    return {**fallback, "_raw": (raw or "")[:240]}


def retrieve(query: str, k: int = 5, config: dict | None = None):
    retriever = vs.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query, config=config or {})
    return docs


def finish_langfuse():
    """兼容旧调用，直接 flush。"""
    langfuse.flush()


def doc_dedupe_key(doc):
    return (doc.metadata.get("chunk_id", doc.metadata.get("doc_id")), doc.page_content[:80])


def show_docs(docs, max_chars: int = 260):
    for i, d in enumerate(docs, 1):
        meta = {
            k: d.metadata.get(k)
            for k in ("type", "source_collection", "parse_source", "chunk_id", "h1", "h2", "h3")
            if k in d.metadata
        }
        print(f"[{i}]", meta)
        print(d.page_content[:max_chars].replace("\n", " "))
        print()


## Langfuse Tips

这个 notebook 既保留了 `print("=== ... trace ===")` 这种课堂版 trace，也把关键步骤上报到了 Langfuse。建议把 notebook 输出和 Langfuse 页面对着看。

### 1. 先看 `Traces`

进入对应的 Langfuse project 后，优先看 `Traces` 页面，并按下面几个维度筛选：
- `session_id`：同一次 notebook 运行会共享一个 `LF_SESSION_ID`
- `user_id`：这里默认是 `rag-notebook-user`
- `tags`：包含 `rag_project`、`agentic_rag`、具体流程标签

初始化 cell 跑完后会打印一个 `langfuse_session`，可以直接拿这个值去搜。

### 2. 再看一条 trace 里面的 observation / span

打开任意一条 trace，重点看：
- 输入问题是什么
- 每一步调用了哪个 LLM / retriever
- 哪一步耗时最高
- 哪一步决定了后续分支
- 最终 answer 是在哪个步骤产出的

### 3. 这个 notebook 里，各条 agent 轨迹通常长什么样

- **Self-RAG**：`self_rag.need_retrieve -> self_rag.retrieve -> self_rag.answer_with_ctx|answer_no_ctx -> self_rag.critique -> self_rag.retrieve_retry -> self_rag.final_answer`
- **CRAG**：`crag.retrieve -> crag.evaluate -> crag.rewrite -> crag.retrieve_retry`
- **Adaptive-RAG**：`adaptive_rag.route` 决定后续分支；可能进入 `retrieve_simple`，也可能进入 `multi_query / retrieve_multi / hyde / retrieve_hyde`，复杂问题则会继续走 `crag(...)`
- **MemoRAG**：`memo_rag.memo_write -> memo_rag.memo_search -> memo_rag.retrieve_with_clue`

### 4. 看 agent 轨迹时，建议重点回答这几个问题

- 路由是不是合理，比如 `simple / medium / complex` 分流对不对
- retrieval 命中的证据是否相关，是否出现明显偏题
- critique / evaluator 是否把“证据不足”和“真正错误”区分开了
- rewrite 之后的第二次检索有没有变好
- memory 命中后，是否真的减少了重复检索

### 5. 一个实用习惯

先看 notebook 里的打印输出，再去 Langfuse 对照同一个 `session_id`。这样既能讲清楚控制流，也能看到真实的调用树、耗时、输入输出和 tags。

## Self-RAG

In [9]:
# 0) Self-RAG（最小闭环）：按需检索 + 对草稿做 critique，不支撑就补检索
# 课件要点：retrieve-on-demand + critique-and-fix

need_retrieve_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的路由器。判断回答这个问题是否需要检索外部知识库。\n"
            "只输出 JSON：{{\"need_retrieve\": 0|1, \"reason\": \"...\"}}。",
        ),
        ("human", "问题：{q}"),
    ]
)

critique_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的 critique 模块。\n"
            "给定问题、草稿答案、以及检索证据（可能为空），判断草稿是否被证据支持。\n"
            "只输出 JSON：{{\"supported\": 0|1, \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- supported=0 时给一个更利于检索的 rewrite（中文）。",
        ),
        ("human", "问题：{q}\n\n草稿：{draft}\n\n证据：\n{evidence}"),
    ]
)

answer_no_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是助手。若缺少证据就保守回答，不要编造。"),
        ("human", "问题：{q}"),
    ]
)

answer_with_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是助手。必须基于给定证据回答；证据不足则明确说不足。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def self_rag(query: str, k: int = 5):
    with lf_observe("self_rag", input={"query": query}):
        print("=== Self-RAG trace ===")
        print("[query]", query)

        raw = llm.invoke(
            need_retrieve_prompt.format_messages(q=query),
            config=lf_config("self_rag.need_retrieve"),
        ).content
        j = parse_json_object(raw, {"need_retrieve": 1, "reason": ""})
        need = 1 if int(j.get("need_retrieve", 1)) == 1 else 0
        reason = j.get("reason", "")
        if "_raw" in j:
            reason = f"bad_json: {j['_raw']}"

        print("[need_retrieve]", need)
        print("[reason]", reason)

        docs = []
        if need:
            docs = retrieve(query, k=k, config=lf_config("self_rag.retrieve"))
            print("[retrieve] k=", k)
            show_docs(docs)

        evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs, 1))

        if docs:
            draft = llm.invoke(
                answer_with_ctx_prompt.format_messages(q=query, evidence=evidence),
                config=lf_config("self_rag.answer_with_ctx"),
            ).content.strip()
        else:
            draft = llm.invoke(
                answer_no_ctx_prompt.format_messages(q=query),
                config=lf_config("self_rag.answer_no_ctx"),
            ).content.strip()

        print("[draft head]", draft[:220].replace("\n", " "))

        raw2 = llm.invoke(
            critique_prompt.format_messages(q=query, draft=draft, evidence=evidence),
            config=lf_config("self_rag.critique"),
        ).content
        c = parse_json_object(raw2, {"supported": 0, "reason": "", "rewrite": ""})
        supported = 1 if int(c.get("supported", 0)) == 1 else 0
        creason = c.get("reason", "")
        rewrite = c.get("rewrite", "")
        if "_raw" in c:
            creason = f"bad_json: {c['_raw']}"

        print("[critique supported]", supported)
        print("[critique reason]", creason)

        if supported:
            langfuse.flush()
            return draft, docs

        if not rewrite:
            rewrite = query
        print("[rewrite]", rewrite)

        docs2 = retrieve(rewrite, k=k, config=lf_config("self_rag.retrieve_retry"))
        print("[retrieve-2] k=", k)
        show_docs(docs2)

        evidence2 = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs2, 1))
        final = llm.invoke(
            answer_with_ctx_prompt.format_messages(q=query, evidence=evidence2),
            config=lf_config("self_rag.final_answer"),
        ).content.strip()
        print("[final head]", final[:220].replace("\n", " "))
        langfuse.flush()
        return final, docs2



self_rag_answer, self_rag_docs = self_rag(Q0, k=50)

=== Self-RAG trace ===
[query] 道通2024年年报里，各产品利润率是多少？
[need_retrieve] 1
[reason] 该问题询问特定公司（道通）在特定年份（2024年）的年报中关于各产品利润率的具体数值。这类具体财务数据通常不会在通用知识库中实时更新，必须检索该公司最新的官方年报或相关财务文档才能获得准确信息。
[retrieve] k= 50
[1] {'parse_source': 'mineru', 'chunk_id': 54, 'h1': '3、深化海外市场本土化建设，品牌影响力持续增强'}
![](images/7a181313e3679ef11afe414ab7b4487617b81ec3c3fa5cbaf92941170e0a485b.jpg)   ![](images/cdd7caaad37371d7b1eae8e28e96cd69b5b628ece2a6b7114b275f3d8fd12c78.jpg)   ![](images/d5780cbb4b6822b6b8933d9b872f132008e2c32aeb633b58306502380316a34e.jpg) 道通“Evergreen”

[2] {'parse_source': 'mineru', 'chunk_id': 811, 'h1': '(一)信用风险'}
信用风险，是指金融工具的一方不能履行义务，造成另一方发生财务损失的风险。

[3] {'parse_source': 'mineru', 'chunk_id': 458, 'h1': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起12个月内的持续经营能力产生重大疑虑的事项或情况。

[4] {'parse_source': 'mineru', 'chunk_id': 836, 'h1': '3、本企业合营和联营企业情况'}
$\surd$ 适用 □不适用   本企业重要的合营或联营企业详见附注十之说明。

[5] {'parse_source': 'mineru', 'chunk_id': 49, 'h1': '（1）车桩云深度融合，打造领先的智能充电网络解决方案'}
![](images/d2d600ea00be02afd055ba9dd88afef4aaa0

## CRAG

In [10]:
# 1) CRAG（Corrective RAG）：先评估检索质量，再决定要不要补救

# 课件要点：Retrieval evaluator -> (refine/search) -> generate
# 这里做课堂版最小闭环：
# - evaluator 只输出 {label, reason, rewrite}
# - label in {correct, ambiguous, incorrect}

CRAGLabel = Literal["correct", "ambiguous", "incorrect"]

crag_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Retrieval Evaluator。\n"
            "输入：用户问题 + 检索到的若干证据片段。\n"
            "输出 JSON：{{\"label\": \"correct|ambiguous|incorrect\", \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- correct: 证据明显能支持回答\n"
            "- ambiguous: 有点相关但不够支撑，需要补检索\n"
            "- incorrect: 基本不相关，需要重写 query 再检索\n"
            "只输出 JSON，不要多余文字。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)

crag_rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Query Rewrite。输出 1 条更利于企业年报检索的中文查询句。不要回答问题。",
        ),
        ("human", "原始问题：{q}\n\n评估原因：{reason}\n\n建议 rewrite：{rewrite}"),
    ]
)


def _crag_inner(query: str, k: int = 5):
    """CRAG 核心逻辑：retrieve → evaluate → (rewrite + retry) → generate answer。"""
    print("=== CRAG trace ===")
    print("[query]", query)

    docs = retrieve(query, k=k, config=lf_config("crag.retrieve"))
    print("[retrieve-1] k=", k)
    show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:350]}" for i, d in enumerate(docs, 1))
    raw = llm.invoke(
        crag_eval_prompt.format_messages(q=query, evidence=evidence),
        config=lf_config("crag.evaluate"),
    ).content

    ev = parse_json_object(raw, {"label": "ambiguous", "reason": "", "rewrite": ""})
    label = ev.get("label", "ambiguous")
    if label not in {"correct", "ambiguous", "incorrect"}:
        label = "ambiguous"
    reason = ev.get("reason", "")
    rewrite_hint = ev.get("rewrite", "")
    if "_raw" in ev:
        reason = f"bad_json: {ev['_raw']}"

    print("[eval]", label)
    print("[reason]", reason)

    final_docs = docs
    if label != "correct":
        rewritten = llm.invoke(
            crag_rewrite_prompt.format_messages(q=query, reason=reason, rewrite=rewrite_hint),
            config=lf_config("crag.rewrite"),
        ).content.strip()
        print("[rewrite]", rewritten)

        final_docs = retrieve(rewritten, k=k, config=lf_config("crag.retrieve_retry"))
        print("[retrieve-2] k=", k)
        show_docs(final_docs)

    final_evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(final_docs, 1))
    answer = llm.invoke(
        answer_with_ctx_prompt.format_messages(q=query, evidence=final_evidence),
        config=lf_config("crag.answer"),
    ).content.strip()
    print("[answer head]", answer[:300].replace("\n", " "))
    return answer, final_docs


def crag(query: str, k: int = 5):
    """独立调用时自建 observation；被 adaptive_rag 嵌套时由外层 context 包裹。"""
    with lf_observe("crag", input={"query": query}):
        answer, docs = _crag_inner(query, k)
        langfuse.flush()
        return answer, docs



crag_answer, crag_docs = crag(Q0, k=50)


=== CRAG trace ===
[query] 道通2024年年报里，各产品利润率是多少？
[retrieve-1] k= 50
[1] {'parse_source': 'mineru', 'chunk_id': 54, 'h1': '3、深化海外市场本土化建设，品牌影响力持续增强'}
![](images/7a181313e3679ef11afe414ab7b4487617b81ec3c3fa5cbaf92941170e0a485b.jpg)   ![](images/cdd7caaad37371d7b1eae8e28e96cd69b5b628ece2a6b7114b275f3d8fd12c78.jpg)   ![](images/d5780cbb4b6822b6b8933d9b872f132008e2c32aeb633b58306502380316a34e.jpg) 道通“Evergreen”

[2] {'parse_source': 'mineru', 'chunk_id': 811, 'h1': '(一)信用风险'}
信用风险，是指金融工具的一方不能履行义务，造成另一方发生财务损失的风险。

[3] {'parse_source': 'mineru', 'chunk_id': 458, 'h1': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起12个月内的持续经营能力产生重大疑虑的事项或情况。

[4] {'parse_source': 'mineru', 'chunk_id': 836, 'h1': '3、本企业合营和联营企业情况'}
$\surd$ 适用 □不适用   本企业重要的合营或联营企业详见附注十之说明。

[5] {'parse_source': 'mineru', 'chunk_id': 49, 'h1': '（1）车桩云深度融合，打造领先的智能充电网络解决方案'}
![](images/d2d600ea00be02afd055ba9dd88afef4aaa0492589d3955851122cd4a50e5cfa.jpg) 充电 APP 智能语音助手   ![](images/d1b24b0aa8fde2d75e3da2f4e483a2112048d789387befaceae004251aea9d3c.j

## Adaptive RAG

In [11]:
# 2) Adaptive-RAG：先判断 query 难度/类型，再决定走轻/重路径
# 课件要点：complexity-aware routing（简单 query 不要走重工作流）

Route = Literal["simple", "medium", "complex"]

route_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Adaptive-RAG 的 router。根据问题复杂度输出 JSON：{{\"route\": \"simple|medium|complex\", \"reason\": \"...\"}}\n"
            "- simple：单事实/单跳，直接检索一次即可\n"
            "- medium：需要更好的召回覆盖（建议 multi-query 或 HyDE）\n"
            "- complex：需要多步/对比/多条件，建议 agentic（例如 CRAG + 多次补检索）\n"
            "只输出 JSON。",
        ),
        ("human", "问题：{q}"),
    ]
)

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Multi-Query 生成器。输出 4 条检索 query，每条一行，不要编号。",
        ),
        ("human", "问题：{q}"),
    ]
)

hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。写一段可能出现在年报中的‘假设答案’，尽量包含可检索关键词，不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)


def adaptive_rag(query: str, k: int = 5):
    with lf_observe("adaptive_rag", input={"query": query}):
        raw = llm.invoke(
            route_prompt.format_messages(q=query),
            config=lf_config("adaptive_rag.route"),
        ).content
        r = parse_json_object(raw, {"route": "medium", "reason": ""})
        route = r.get("route", "medium")
        if route not in {"simple", "medium", "complex"}:
            route = "medium"
        reason = r.get("reason", "")
        if "_raw" in r:
            reason = f"bad_json: {r['_raw']}"

        print("=== Adaptive-RAG trace ===")
        print("[query]", query)
        print("[route]", route)
        print("[reason]", reason)

        if route == "simple":
            docs = retrieve(query, k=k, config=lf_config("adaptive_rag.retrieve_simple"))
            show_docs(docs)
            final_docs = docs

        elif route == "medium":
            queries = [
                s.strip()
                for s in llm.invoke(
                    multi_prompt.format_messages(q=query),
                    config=lf_config("adaptive_rag.multi_query"),
                ).content.splitlines()
                if s.strip()
            ]
            if not queries:
                queries = [query]
            print("[multi-query]", queries)
            seen, merged = set(), []
            for q in queries:
                for d in retrieve(q, k=3, config=lf_config("adaptive_rag.retrieve_multi")):
                    key = doc_dedupe_key(d)
                    if key not in seen:
                        merged.append(d)
                        seen.add(key)
            print("[multi-query merged]", len(merged))
            show_docs(merged[:8])

            hypo = llm.invoke(
                hyde_prompt.format_messages(q=query),
                config=lf_config("adaptive_rag.hyde"),
            ).content.strip()
            print("[hyde hypo head]", hypo[:200].replace("\n", " "))
            docs_h = retrieve(hypo, k=k, config=lf_config("adaptive_rag.retrieve_hyde"))
            print("[hyde hits]")
            show_docs(docs_h)
            final_docs = merged[:8] if merged else docs_h

        else:
            # complex -> CRAG
            crag_answer, final_docs = _crag_inner(query, k=k)
            print("[crag answer head]", crag_answer[:200].replace("\n", " "))

        evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(final_docs, 1))
        answer = llm.invoke(
            answer_with_ctx_prompt.format_messages(q=query, evidence=evidence),
            config=lf_config("adaptive_rag.answer"),
        ).content.strip()
        print("[answer head]", answer[:300].replace("\n", " "))
        langfuse.flush()
        return answer, final_docs



adaptive_answer, adaptive_docs = adaptive_rag(Q0, k=50)


=== Adaptive-RAG trace ===
[query] 道通2024年年报里，各产品利润率是多少？
[route] medium
[reason] 问题要求查询特定公司（道通）在特定年份（2024年）的年报中，各产品的利润率数据。这需要从年报中提取并汇总多个产品的财务信息，属于需要较好召回覆盖的多事实查询，建议使用 multi-query 或 HyDE 来确保检索到所有相关产品数据。
[multi-query] ['道通科技2024年产品毛利率', '道通科技2024年各业务线利润', '道通科技2024年报分产品盈利情况', '道通科技2024年主营业务利润率']
[multi-query merged] 12
[1] {'parse_source': 'mineru', 'chunk_id': 54, 'h1': '3、深化海外市场本土化建设，品牌影响力持续增强'}
![](images/7a181313e3679ef11afe414ab7b4487617b81ec3c3fa5cbaf92941170e0a485b.jpg)   ![](images/cdd7caaad37371d7b1eae8e28e96cd69b5b628ece2a6b7114b275f3d8fd12c78.jpg)   ![](images/d5780cbb4b6822b6b8933d9b872f132008e2c32aeb633b58306502380316a34e.jpg) 道通“Evergreen”

[2] {'parse_source': 'mineru', 'chunk_id': 458, 'h1': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起12个月内的持续经营能力产生重大疑虑的事项或情况。

[3] {'parse_source': 'mineru', 'chunk_id': 704, 'h1': '44、 其他流动负债'}
其他流动负债情况   √适用 □不适用   单位：元币种：人民币   | 项目 | 期末余额 | 期初余额 | | --- | --- | --- | | 质保金 | 23,598,210.79 | | | 待转销项税额 | 3,776,316.29 | 594,036.63 | | 合计 | 27,374,5

## MemoRAG

In [ ]:
# 3) MemoRAG（课堂版）：把“已回答过的高价值线索/答案片段”存入 memory store
# 思路：
# - memory 是一个独立的向量库（Chroma collection），内容是：{question, answer_clue, citations}
# - 新问题先查 memory；若命中高相似，就直接复用线索并补少量检索

MEMO_COLLECTION = "autel_annual_report_2024_memo"
memo_vs = Chroma(
    collection_name=MEMO_COLLECTION,
    embedding_function=emb,
    persist_directory=str(CHROMA_DIR),
)

memo_clue_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 MemoRAG 的 memory writer。\n"
            "给定问题 + 证据片段，提炼 5-8 条可复用的 Answer Clues（要像索引关键词），不要编造数值。\n"
            "输出 JSON：{{\"clues\": [\"...\"], \"summary\": \"...\"}}。只输出 JSON。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def memo_write(question: str, docs, memo_id: str | None = None):
    evidence = "\n\n".join(d.page_content[:500] for d in docs[:5])
    raw = llm.invoke(
        memo_clue_prompt.format_messages(q=question, evidence=evidence),
        config=lf_config("memo_rag.memo_write", ["memo_rag", "memory_write"]),
    ).content
    j = parse_json_object(raw, {"clues": [], "summary": raw[:500]})

    clues = j.get("clues", [])
    if not isinstance(clues, list):
        clues = []
    summary = j.get("summary", raw[:500])

    content = "\n".join(["Answer clues:"] + clues + ["", "Summary:", summary])
    meta = {"type": "memo", "source_collection": COLLECTION}

    memo_id = memo_id or f"memo-{hashlib.sha1(question.encode('utf-8')).hexdigest()[:16]}"
    existing = memo_vs.get(ids=[memo_id], include=[])
    if existing and existing.get("ids"):
        memo_vs.delete(ids=[memo_id])

    memo_vs.add_texts([content], ids=[memo_id], metadatas=[meta])
    memo_vs.persist()
    return memo_id


def memo_search(question: str, k: int = 3):
    retriever = memo_vs.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(question, config=lf_config("memo_rag.memo_search", ["memo_rag", "memory_search"]))


# 先用一个问题跑 CRAG，然后把结果写入 memo
seed_q = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？"
_, seed_docs = crag(seed_q, k=10)
mid = memo_write(seed_q, seed_docs)
print("[memo saved]", mid)

# 再问一个相近问题，先查 memo
follow_q = "道通年报里各产品线收入占比/结构怎么描述？"
mem_hits = memo_search(follow_q, k=3)
print("=== MemoRAG trace ===")
print("[query]", follow_q)
print("[memo hits]")
show_docs(mem_hits, max_chars=400)

# 如果 memo 命中，再用 memo 的线索补一次轻检索，然后生成答案
if mem_hits:
    memo_query = mem_hits[0].page_content
    print("[retrieve with memo clue]")
    docs = retrieve(memo_query, k=10, config=lf_config("memo_rag.retrieve_with_clue"))
    show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs, 1))
    memo_answer = llm.invoke(
        answer_with_ctx_prompt.format_messages(q=follow_q, evidence=evidence),
        config=lf_config("memo_rag.answer"),
    ).content.strip()
    print("[memo answer head]", memo_answer[:300].replace("\n", " "))

langfuse.flush()


=== CRAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？
[retrieve-1] k= 5
[1] {'parse_source': 'mineru', 'chunk_id': 54, 'h1': '3、深化海外市场本土化建设，品牌影响力持续增强'}
![](images/7a181313e3679ef11afe414ab7b4487617b81ec3c3fa5cbaf92941170e0a485b.jpg)   ![](images/cdd7caaad37371d7b1eae8e28e96cd69b5b628ece2a6b7114b275f3d8fd12c78.jpg)   ![](images/d5780cbb4b6822b6b8933d9b872f132008e2c32aeb633b58306502380316a34e.jpg) 道通“Evergreen”

[2] {'parse_source': 'mineru', 'chunk_id': 519, 'h1': '(4)房租及管理费'}
房租及管理费主要核算为研发活动而发生的物业管理及水电费。

[3] {'parse_source': 'mineru', 'chunk_id': 458, 'h1': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起12个月内的持续经营能力产生重大疑虑的事项或情况。

[4] {'parse_source': 'mineru', 'chunk_id': 883, 'h1': '(1). 报告分部的确定依据与会计政策'}
$\surd$ 适用 □不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[5] {'parse_source': 'mineru', 'chunk_id': 460, 'h1': '1、遵循企业会计准则的声明'}
本公司所编制的财务报表符合企业会计准则的要求，真实、完整地反映了公司的财务状况、经营成果和现金流量等有关信息。

[eval] incorrect
[reason] 提供的证据片段与用户询问的'主营业务/产品线的收入结构'完全不相关。证据[1

/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_41197/2295889009.py:48: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  memo_vs.persist()


[memo saved] memo-dea68d5112dc3582
=== MemoRAG trace ===
[query] 道通年报里各产品线收入占比/结构怎么描述？
[memo hits]
[1] {'type': 'memo', 'source_collection': 'autel_annual_report_2024'}
Answer clues: 持续经营基础 营业周期较短 12个月流动性划分 未履行履约义务收入 499,395,989.55元 未来12-36个月确认 用户使用确认收入  Summary: 证据片段主要说明财务报表编制基础和收入确认情况，未提供主营业务/产品线的具体收入结构信息。

[retrieve with memo clue]
[1] {'parse_source': 'mineru', 'chunk_id': 733, 'h1': '(4). 分摊至剩余履约义务的说明'}
√适用 □不适用   本报告期末已签订合同、但尚未履行或尚未履行完毕的履约义务所对应的收入金额为499,395,989.55元，公司预计该金额将随着用户的使用，在未来12-36个月内确认为收入。

[2] {'parse_source': 'mineru', 'chunk_id': 462, 'h1': '3、营业周期'}
√适用 □不适用   公司经营业务的营业周期较短，以12个月作为资产和负债的流动性划分标准。

[3] {'parse_source': 'mineru', 'chunk_id': 883, 'h1': '(1). 报告分部的确定依据与会计政策'}
$\surd$ 适用 □不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[4] {'parse_source': 'mineru', 'chunk_id': 127, 'h1': '2、存货规模较高的风险'}
报告期末，公司存货净额为11.51亿元，占流动资产的比例为 $2 7 . 1 2 \%$ 。若未来原材料价格大幅波动，或产品市场价格大幅下跌，公司存货将面临跌价损失风险。

[5] {'parse_source': 'mineru', 'chunk_id': 



如果页面里 trace 比较多，可以优先搜这些字段：
- 当前 notebook 运行打印出来的 `langfuse_session`
- `trace name`，例如 `self_rag.need_retrieve`、`crag.evaluate`、`adaptive_rag.route`
- `tags`，例如 `self_rag`、`crag`、`adaptive_rag`、`memo_rag`



# 课后总结


- **CRAG**：
  - 先检索一次
  - LLM 只负责当“裁判”：这批证据够不够
  - 不够就触发 rewrite + 补检索
- **Adaptive-RAG**：
  - 核心是 *routing*：简单问题不要走重链路
  - 这节用 `simple/medium/complex` 三档让逻辑一眼可见
- **MemoRAG**：
  - 把“已发现的线索”当作新的可检索资产
  - 后续相似问题先查 memo，减少重复检索
